# 🚀 02 - Fine-Tuning del Modelo de Lenguaje (RotBot English Coach)

Este notebook guía el proceso de subida del dataset y lanzamiento del trabajo de **Supervised Fine-Tuning (SFT)**.

### Plataformas Soportadas:
1. **Google AI Studio (Gemini 1.5 Flash / Gemini Pro):** Recomendado para alta velocidad, contexto largo y optimización en costo.
2. **OpenAI (GPT-4o-mini / GPT-3.5-turbo):** Alternativa para fine-tuning estándar con ChatML.

In [ ]:
# 1. Configuración de dependencias y variables de entorno
import os
import sys
import time
import json
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Cargar .env
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

TRAIN_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "train.jsonl")
VAL_JSONL_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "val.jsonl")

print(f"Directorio del proyecto: {PROJECT_ROOT}")
print(f"Train file exists: {os.path.exists(TRAIN_JSONL_PATH)}")
print(f"Val file exists:   {os.path.exists(VAL_JSONL_PATH)}")

## 2. Opción A: Fine-Tuning con Google Gemini (AI Studio)
Utilizamos la API de Google Generative AI para ajustar un modelo base (ej. `models/gemini-1.5-flash-001-tuning`).

In [ ]:
import google.generativeai as genai

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key or gemini_api_key == "your_gemini_api_key_here":
    print("⚠️ Por favor, define GEMINI_API_KEY en tu archivo .env antes de continuar.")
else:
    genai.configure(api_key=gemini_api_key)
    print("✅ Gemini API configurada con éxito.")
    
    # Cargar datos de entrenamiento en formato lista/dict para Gemini Tuning
    training_data = []
    with open(TRAIN_JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if "messages" in record:
                    # Extraer texto de usuario y asistente
                    u_text = next((m["content"] for m in record["messages"] if m["role"] == "user"), "")
                    a_text = next((m["content"] for m in record["messages"] if m["role"] == "assistant"), "")
                    if u_text and a_text:
                        training_data.append({"text_input": u_text, "output": a_text})
                        
    print(f"Total de ejemplos preparados para Gemini: {len(training_data)}")

In [ ]:
# Lanzamiento del Job en Google AI Studio (Descomentar para ejecutar)
"""
operation = genai.create_tuned_model(
    source_model="models/gemini-1.5-flash-001-tuning",
    training_data=training_data,
    id="rotbot-english-coach-v1",
    display_name="RotBot English Coach v1",
    description="Fine-tuned model for Spanish speakers learning English",
    epoch_count=10,
    batch_size=4,
    learning_rate=0.001,
)

print(f"🚀 Operación iniciada: {operation.name}")
for status in operation.wait_bar():
    time.sleep(10)

result_model = operation.result()
print(f"✅ Modelo entrenado exitosamente: {result_model.name}")
"""

## 3. Opción B: Fine-Tuning con OpenAI (GPT-4o-mini / GPT-3.5-turbo)
Subida directa de `train.jsonl` y `val.jsonl` a OpenAI Files API y creación del trabajo de fine-tuning.

In [ ]:
from openai import OpenAI

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key or openai_api_key.startswith("your_"):
    print("ℹ️ OPENAI_API_KEY no configurada. Si usas OpenAI, agrégala en .env.")
else:
    client = OpenAI(api_key=openai_api_key)
    print("✅ Cliente OpenAI inicializado.")
    
    # Subida de archivo de entrenamiento
    print("Subiendo train.jsonl...")
    with open(TRAIN_JSONL_PATH, "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    print(f"✅ Training File ID: {train_file.id}")
    
    # Subida de archivo de validación
    print("Subiendo val.jsonl...")
    with open(VAL_JSONL_PATH, "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")
    print(f"✅ Validation File ID: {val_file.id}")

In [ ]:
# Lanzamiento del Fine-Tuning Job en OpenAI (Descomentar para ejecutar)
"""
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="rotbot-coach-v1",
    hyperparameters={
        "n_epochs": 3
    }
)
print(f"🚀 Fine-tuning job creado con ID: {job.id}")
"""

## 4. Guardar ID del Modelo Entrenado
Una vez completado el entrenamiento, guarda el identificador en tu `.env` bajo `TUNED_MODEL_ID` para evaluarlo en el siguiente notebook (`03_chat_evaluation.ipynb`).